# Lecture 1: The Storage and Serialization Bridge

## 1. Escaping the Small Files Problem
When dealing with millions of small files (like individual images), the I/O latency equation for independent files is dominated by seek time.
Let $N$ track total files, $S_{file}$ track average file size, $T_{seek}$ track storage latency, and $B_{network}$ track bandwidth:
$$T_{total\_independent} = N \times \left(T_{seek} + \frac{S_{file}}{B_{network}}\right)$$

The amortized equation for contiguous sharding (like WebDataset Tar archives chunked by thousands) is:
$$T_{total\_sharded} \approx T_{seek\_archive} + \frac{N \times S_{file}}{B_{network}}$$

Use the simulator below to dynamically observe how standard HTTP roundtrips destroy GPU throughput when accessing independent files compared to streaming chunks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def plot_io_latency(N=60000, T_seek_ms=50.0, S_file_kb=150.0, B_network_mbps=1000.0):
    T_seek = T_seek_ms / 1000.0
    S_file = S_file_kb / 1024.0   # MB
    B_nw = B_network_mbps / 8.0   # MB/s

    # Baseline (Independent Files)
    time_seek_baseline = N * T_seek
    time_transfer = N * (S_file / B_nw)
    total_baseline = time_seek_baseline + time_transfer

    # Amortized (WebDataset Tar Archives, eg 1000 items per tar)
    shards = max(1, N // 1000)
    time_seek_amortized = shards * T_seek
    total_amortized = time_seek_amortized + time_transfer

    fig, ax = plt.subplots(figsize=(10, 5))
    categories = ['Independent Files', 'Contiguous Shards']
    seek_times = [time_seek_baseline, time_seek_amortized]
    transfer_times = [time_transfer, time_transfer]

    ax.barh(categories, transfer_times, label='Transfer Time (Bandwidth)', color='#3498db')
    ax.barh(categories, seek_times, left=transfer_times, label='Seek Time Penalty', color='#e74c3c')
    
    ax.set_xlabel('Total Time (Seconds)')
    ax.set_title(f'I/O Bottleneck Simulator | Streaming handles standard data {total_baseline / total_amortized:.1f}x Faster')
    ax.legend()
    plt.tight_layout()
    plt.show()

latency_ui = widgets.interactive(
    plot_io_latency,
    N=widgets.IntSlider(min=1000, max=1000000, step=1000, value=60000, description='Total Files:'),
    T_seek_ms=widgets.FloatSlider(min=1.0, max=200.0, step=1.0, value=50.0, description='Seek (ms):'),
    S_file_kb=widgets.FloatSlider(min=10.0, max=5000.0, step=10.0, value=150.0, description='Size (KB):'),
    B_network_mbps=widgets.FloatSlider(min=100.0, max=10000.0, step=100.0, value=1000.0, description='BW (Mbps):')
)
display(latency_ui)

## 2. Interactive S3 Exploration
We can verify that our generated shards are correctly stored in our simulated S3 (MinIO) bucket using `boto3` pagination.

In [ ]:
import boto3

# Initialize S3 Client matching our MinIO configuration
s3 = boto3.client(
    's3',
    endpoint_url='http://127.0.0.1:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin'
)

# Use pagination to list objects in the cifar-streaming bucket
paginator = s3.get_paginator('list_objects_v2')
try:
    pages = paginator.paginate(Bucket='cifar-streaming')
    print("Files in 'cifar-streaming':")
    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents'][:5]: # Print first 5
                print(f"- {obj['Key']} ({obj['Size'] / 1024 / 1024:.2f} MB)")
            print("... (truncated)")
except Exception as e:
    print(f"Error accessing bucket: {e}")

## 3. Streaming Data Pipeline
Now we build our `WebDataset` pipeline to read directly from the streaming endpoint. We employ an interactive widget to iterate through the stream seamlessly!

In [ ]:
import webdataset as wds
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# The URL prefix to our local MinIO shards.
url = "http://127.0.0.1:9000/cifar-streaming/cifar-train-{000000..000049}.tar"

# Construct WebDataset pipeline, suppressing warning with shardshuffle=False
dataset = (
    wds.WebDataset(url, shardshuffle=False)
    .decode("rgb")
    .to_tuple("png", "cls")
)

# Fetch 100 images into a local buffer for the interactive widget
buffer_images, buffer_labels = [], []

try:
    for idx, (img, label) in enumerate(dataset):
        buffer_images.append(img)
        buffer_labels.append(label)
        if idx == 99:
            break

    def view_image(index=0):
        plt.figure(figsize=(4, 4))
        plt.imshow(buffer_images[index])
        plt.title(f"Class Label: {buffer_labels[index]} | Stream Index: {index}")
        plt.axis('off')
        plt.show()

    if buffer_images:
        slider = widgets.IntSlider(min=0, max=len(buffer_images)-1, value=0, description='Stream Index:')
        display(widgets.interactive(view_image, index=slider))
except Exception as e:
    print(f"Could not read from MinIO stream. {e}")